# gs_hmd completed survival curves with continuation markers

This notebook reads the local merged histories in `outputs/run_data/gs_hmd`, plots the completed survival curves, and marks the point where a run stopped before its continuation was merged locally.

In [1]:
from pathlib import Path
import json
import re
import tomllib

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


def find_task_dir(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file():
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
RUN_DATA_GROUP_DIR = TASK_DIR / "outputs" / "run_data" / "gs_hmd"
RUNS_ROOT = RUN_DATA_GROUP_DIR / "runs"
MERGE_REPORT_DIR = RUN_DATA_GROUP_DIR / "local_rescue_merges"
CONFIG_DIR = TASK_DIR / "configs" / "gnn_graph_screening" / "heterogenous_message_direction"
FULL_TEST_EVAL_DIR = TASK_DIR / "outputs" / "full_test_eval"
CONTINUATION_EVAL_DIR = FULL_TEST_EVAL_DIR / "continuation_20260731"

print("task dir:", TASK_DIR)
print("run data:", RUN_DATA_GROUP_DIR)
print("configs:", CONFIG_DIR)
print("old full-test evals:", FULL_TEST_EVAL_DIR)
print("continuation evals:", CONTINUATION_EVAL_DIR)

task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
run data: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_hmd
configs: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/gnn_graph_screening/heterogenous_message_direction
old full-test evals: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval
continuation evals: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval/continuation_20260731


## Controls

In [2]:
# Change this to plot a different survival signal.
METRIC = "test/charts/episodic_survival"
# Useful alternatives:
# METRIC = "train_eval/charts/episodic_survival"
# METRIC = "test/episodic_survival"
# METRIC = "train_eval/episodic_survival"

SMOOTH_WINDOW = 5
VALUE_SCALE = 100.0
FIG_WIDTH = 1500
FIG_HEIGHT = 760
SHOW_SEED_CURVES = True
SHOW_MEAN_CURVES = True

DIRECTION_CODES = {
    "bidirectional": "bi",
    "asset_to_busbar": "a2b",
    "busbar_to_asset": "b2a",
}
DIRECTION_ORDER = ["bi", "a2b", "b2a"]

DIRECTION_COLORS = {
    "g=bi / l=bi": "#5b6472",
    "g=bi / l=a2b": "#1f77b4",
    "g=bi / l=b2a": "#00a3ad",
    "g=a2b / l=bi": "#2ca02c",
    "g=a2b / l=a2b": "#d62728",
    "g=a2b / l=b2a": "#ff7f0e",
    "g=b2a / l=bi": "#7b61ff",
    "g=b2a / l=a2b": "#8c564b",
    "g=b2a / l=b2a": "#e377c2",
}

print("metric:", METRIC)

metric: test/charts/episodic_survival


## Build the run catalog

In [3]:
def direction_code(value):
    return DIRECTION_CODES.get(str(value), str(value))


def config_record(path):
    with path.open("rb") as file:
        config = tomllib.load(file)
    args = config.get("args", {})
    run = config.get("run", {})
    generator_direction = str(args.get("gnn_generator_edge_direction", "bidirectional"))
    load_direction = str(args.get("gnn_load_edge_direction", "bidirectional"))
    generator_code = direction_code(generator_direction)
    load_code = direction_code(load_direction)
    return {
        "config_path": str(path.relative_to(TASK_DIR)),
        "config": path.name,
        "run_name": str(run.get("name") or args.get("exp_tag") or path.stem),
        "seed": int(args.get("seed", 0)),
        "generator_direction": generator_direction,
        "load_direction": load_direction,
        "line_direction": str(args.get("gnn_line_node_edge_direction", "bidirectional")),
        "summary_direction": str(args.get("gnn_summary_edge_direction", "bidirectional")),
        "generator_code": generator_code,
        "load_code": load_code,
        "direction_label": f"g={generator_code} / l={load_code}",
        "configured_steps": int(args.get("total_timesteps", 15_000_000)),
    }


if not CONFIG_DIR.exists():
    raise FileNotFoundError(f"Missing config folder: {CONFIG_DIR}")

config_catalog = pd.DataFrame([config_record(path) for path in sorted(CONFIG_DIR.glob("*.toml"))])
if config_catalog.empty:
    raise RuntimeError(f"No TOML files found in {CONFIG_DIR}")

config_catalog["direction_label"] = pd.Categorical(
    config_catalog["direction_label"],
    categories=[f"g={g} / l={l}" for g in DIRECTION_ORDER for l in DIRECTION_ORDER],
    ordered=True,
)

print(f"Cataloged {len(config_catalog)} declared runs.")
display(
    config_catalog.pivot_table(
        index="generator_code",
        columns="load_code",
        values="seed",
        aggfunc="count",
        fill_value=0,
    ).reindex(index=DIRECTION_ORDER, columns=DIRECTION_ORDER)
)

Cataloged 27 declared runs.


load_code,bi,a2b,b2a
generator_code,,,
bi,3,3,3
a2b,3,3,3
b2a,3,3,3


## Load local merged histories

In [4]:
def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def run_name_from_dir(run_dir):
    metadata = read_json(run_dir / "metadata.json")
    return str(metadata.get("name") or metadata.get("run_name") or run_dir.name.split("__", 1)[0]).strip()


def run_id_from_dir(run_dir):
    metadata = read_json(run_dir / "metadata.json")
    return str(metadata.get("id") or metadata.get("run_id") or run_dir.name.rsplit("__", 1)[-1]).strip()


def read_history(run_dir):
    parquet_path = run_dir / "history.parquet"
    csv_path = run_dir / "history.csv.gz"
    if parquet_path.exists():
        history = pd.read_parquet(parquet_path)
    elif csv_path.exists():
        history = pd.read_csv(csv_path)
    else:
        return None
    run_name = run_name_from_dir(run_dir)
    run_id = run_id_from_dir(run_dir)
    if "_step" not in history.columns:
        if "step" in history.columns:
            history = history.rename(columns={"step": "_step"})
        else:
            history.insert(0, "_step", range(len(history)))
    history["_step"] = pd.to_numeric(history["_step"], errors="coerce")
    history = history.dropna(subset=["_step"]).sort_values("_step").reset_index(drop=True)
    history["run_name"] = run_name
    history["run_id"] = run_id
    return history


frames = []
for run_dir in sorted(path for path in RUNS_ROOT.iterdir() if path.is_dir()):
    run_name = run_name_from_dir(run_dir)
    if run_name not in set(config_catalog["run_name"]):
        continue
    history = read_history(run_dir)
    if history is not None:
        frames.append(history)

if not frames:
    raise RuntimeError(f"No local histories found under {RUNS_ROOT}")

history_df = pd.concat(frames, ignore_index=True, sort=False)
history_df = history_df.merge(
    config_catalog[
        [
            "run_name",
            "seed",
            "generator_code",
            "load_code",
            "direction_label",
            "configured_steps",
        ]
    ],
    on="run_name",
    how="left",
)
history_df["step_m"] = history_df["_step"] / 1_000_000

print(f"Loaded {history_df['run_name'].nunique()} runs and {len(history_df):,} history rows.")
print("Columns containing survival:")
print("\n".join(sorted(col for col in history_df.columns if "episodic_survival" in col)))

Loaded 27 runs and 9,711 history rows.
Columns containing survival:
test/charts/episodic_survival
test/episodic_survival
train_eval/charts/episodic_survival
train_eval/episodic_survival


## Load discontinuation points

In [5]:
def latest_merge_report_path():
    reports = sorted(MERGE_REPORT_DIR.glob("merge_report_*.csv"))
    if not reports:
        return None
    return reports[-1]


MERGE_REPORT_PATH = latest_merge_report_path()
if MERGE_REPORT_PATH is None:
    discontinuity_df = pd.DataFrame(
        columns=["run_name", "old_max_step", "rescue_min_step", "rescue_max_step"]
    )
    print("No merge report found. Discontinuation markers will be empty.")
else:
    discontinuity_df = pd.read_csv(MERGE_REPORT_PATH)
    for column in ["old_max_step", "rescue_min_step", "rescue_max_step"]:
        discontinuity_df[column] = pd.to_numeric(discontinuity_df[column], errors="coerce")
    discontinuity_df["discontinuity_step"] = discontinuity_df["old_max_step"]
    discontinuity_df["resume_step"] = discontinuity_df["rescue_min_step"]
    discontinuity_df["discontinuity_step_m"] = discontinuity_df["discontinuity_step"] / 1_000_000
    discontinuity_df["resume_step_m"] = discontinuity_df["resume_step"] / 1_000_000
    discontinuity_df = discontinuity_df.merge(
        config_catalog[["run_name", "seed", "direction_label"]],
        on="run_name",
        how="left",
    )
    print("merge report:", MERGE_REPORT_PATH)
    display(
        discontinuity_df[
            [
                "direction_label",
                "seed",
                "run_name",
                "discontinuity_step_m",
                "resume_step_m",
            ]
        ].sort_values(["direction_label", "seed"]).round(3)
    )

merge report: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_hmd/local_rescue_merges/merge_report_20260731T052824Z.csv


,direction_label,seed,run_name,discontinuity_step_m,resume_step_m
11,g=bi / l=bi,0,gs_hmd_hetero_n0_none_gbi_lbi_s0,13.271,13.313
12,g=bi / l=bi,2,gs_hmd_hetero_n0_none_gbi_lbi_s2,13.188,13.230
9,g=bi / l=a2b,0,gs_hmd_hetero_n0_none_gbi_la2b_s0,12.773,12.815
10,g=bi / l=b2a,0,gs_hmd_hetero_n0_none_gbi_lb2a_s0,13.893,13.935
3,g=a2b / l=bi,0,gs_hmd_hetero_n0_none_ga2b_lbi_s0,12.939,12.981
0,g=a2b / l=a2b,0,gs_hmd_hetero_n0_none_ga2b_la2b_s0,14.018,14.059
1,g=a2b / l=b2a,0,gs_hmd_hetero_n0_none_ga2b_lb2a_s0,12.276,12.317
2,g=a2b / l=b2a,1,gs_hmd_hetero_n0_none_ga2b_lb2a_s1,13.935,13.976
8,g=b2a / l=bi,1,gs_hmd_hetero_n0_none_gb2a_lbi_s1,11.944,11.985
4,g=b2a / l=a2b,1,gs_hmd_hetero_n0_none_gb2a_la2b_s1,13.769,13.810


## Coverage after local merge

In [6]:
coverage = (
    history_df.groupby("run_name", as_index=False)
    .agg(rows=("_step", "size"), max_step=("_step", "max"))
    .merge(config_catalog, on="run_name", how="left")
)
coverage["max_step_m"] = coverage["max_step"] / 1_000_000
coverage["completion_pct"] = 100 * coverage["max_step"] / coverage["configured_steps"]
coverage = coverage.merge(
    discontinuity_df[["run_name", "discontinuity_step_m", "resume_step_m"]],
    on="run_name",
    how="left",
)
coverage["was_locally_merged"] = coverage["discontinuity_step_m"].notna()

with pd.option_context("display.max_colwidth", None):
    display(
        coverage.sort_values(["direction_label", "seed"])[
            [
                "direction_label",
                "seed",
                "run_name",
                "rows",
                "max_step_m",
                "completion_pct",
                "was_locally_merged",
                "discontinuity_step_m",
                "resume_step_m",
            ]
        ].round(3)
    )

,direction_label,seed,run_name,rows,max_step_m,completion_pct,was_locally_merged,discontinuity_step_m,resume_step_m
24,g=bi / l=bi,0,gs_hmd_hetero_n0_none_gbi_lbi_s0,361,14.971,99.809,True,13.271,13.313
25,g=bi / l=bi,1,gs_hmd_hetero_n0_none_gbi_lbi_s1,358,14.847,98.980,False,NaN,NaN
26,g=bi / l=bi,2,gs_hmd_hetero_n0_none_gbi_lbi_s2,361,14.971,99.809,True,13.188,13.230
18,g=bi / l=a2b,0,gs_hmd_hetero_n0_none_gbi_la2b_s0,361,14.971,99.809,True,12.773,12.815
19,g=bi / l=a2b,1,gs_hmd_hetero_n0_none_gbi_la2b_s1,348,14.432,96.215,False,NaN,NaN
20,g=bi / l=a2b,2,gs_hmd_hetero_n0_none_gbi_la2b_s2,361,14.971,99.809,False,NaN,NaN
21,g=bi / l=b2a,0,gs_hmd_hetero_n0_none_gbi_lb2a_s0,361,14.971,99.809,True,13.893,13.935
22,g=bi / l=b2a,1,gs_hmd_hetero_n0_none_gbi_lb2a_s1,350,14.515,96.768,False,NaN,NaN
23,g=bi / l=b2a,2,gs_hmd_hetero_n0_none_gbi_lb2a_s2,361,14.971,99.809,False,NaN,NaN
6,g=a2b / l=bi,0,gs_hmd_hetero_n0_none_ga2b_lbi_s0,361,14.971,99.809,True,12.939,12.981


## Old vs continuation full-test evaluations

This compares the previous full-test evaluation JSONs against the continuation best-checkpoint evaluations. The old files remain in `outputs/full_test_eval`; the new files are read from `outputs/full_test_eval/continuation_20260731`.

In [7]:
def best_checkpoint_run_name(record, result_path=None):
    checkpoint_stem = Path(str(record.get("checkpoint", ""))).stem
    if not checkpoint_stem and result_path is not None:
        checkpoint_stem = result_path.stem
    prefix = "best_test_"
    if checkpoint_stem.startswith(prefix):
        checkpoint_stem = checkpoint_stem[len(prefix):]
    # Fallback for default output-json filenames when the checkpoint field is missing.
    checkpoint_stem = re.sub(r"_step\d+_job[^_]+$", "", checkpoint_stem)
    return checkpoint_stem


def load_full_test_results(eval_dir, source_label, recursive=False):
    paths = sorted(
        eval_dir.rglob("best_test_gs_hmd_*.json")
        if recursive and eval_dir.exists()
        else eval_dir.glob("best_test_gs_hmd_*.json")
        if eval_dir.exists()
        else []
    )
    rows = []
    run_name_set = set(config_catalog["run_name"])
    for result_path in paths:
        with result_path.open("r", encoding="utf-8") as file:
            record = json.load(file)
        run_name = best_checkpoint_run_name(record, result_path=result_path)
        if run_name not in run_name_set:
            continue
        rows.append(
            {
                "run_name": run_name,
                "eval_source": source_label,
                "eval_step": int(record["checkpoint_global_step"]),
                "eval_step_m": int(record["checkpoint_global_step"]) / 1_000_000,
                "survival_pct": float(record["survival_percent"]),
                "eval_episodes": int(record["eval_episodes"]),
                "eval_split": str(record["split"]),
                "eval_deterministic": bool(record["deterministic_eval"]),
                "created_at": record.get("created_at", ""),
                "eval_json": str(result_path.relative_to(TASK_DIR)),
            }
        )
    frame = pd.DataFrame(rows)
    if frame.empty:
        print(f"No {source_label} full-test JSONs found under {eval_dir}")
        return frame
    frame["created_at_ts"] = pd.to_datetime(frame["created_at"], errors="coerce", utc=True)
    duplicated = frame[frame.duplicated("run_name", keep=False)]
    if not duplicated.empty:
        print(
            f"Warning: {source_label} has duplicate run results; keeping the latest "
            "created_at / highest-step file per run."
        )
        frame = (
            frame.sort_values(["run_name", "created_at_ts", "eval_step", "eval_json"])
            .drop_duplicates("run_name", keep="last")
            .reset_index(drop=True)
        )
    return frame.drop(columns=["created_at_ts"])


old_full_test_df = load_full_test_results(FULL_TEST_EVAL_DIR, "old", recursive=False)
new_full_test_df = load_full_test_results(CONTINUATION_EVAL_DIR, "continuation", recursive=False)
print(f"old evaluations: {len(old_full_test_df)}")
print(f"continuation evaluations: {len(new_full_test_df)}")

comparison_columns = [
    "run_name",
    "eval_step_m",
    "survival_pct",
    "eval_episodes",
    "eval_split",
    "eval_deterministic",
    "eval_json",
]
eval_compare_df = old_full_test_df[comparison_columns].merge(
    new_full_test_df[comparison_columns],
    on="run_name",
    how="outer",
    suffixes=("_old", "_new"),
)
eval_compare_df = eval_compare_df.merge(
    config_catalog[
        ["run_name", "seed", "generator_code", "load_code", "direction_label"]
    ],
    on="run_name",
    how="left",
)
eval_compare_df["survival_delta_pp"] = (
    eval_compare_df["survival_pct_new"] - eval_compare_df["survival_pct_old"]
)
eval_compare_df["step_delta_m"] = (
    eval_compare_df["eval_step_m_new"] - eval_compare_df["eval_step_m_old"]
)

updated_eval_df = eval_compare_df.dropna(
    subset=["survival_pct_old", "survival_pct_new"]
).copy()
print(f"matched old/new evaluations: {len(updated_eval_df)}")
with pd.option_context("display.max_colwidth", None):
    display(
        updated_eval_df.sort_values(["direction_label", "seed"])[
            [
                "direction_label",
                "seed",
                "run_name",
                "eval_step_m_old",
                "survival_pct_old",
                "eval_step_m_new",
                "survival_pct_new",
                "survival_delta_pp",
                "step_delta_m",
                "eval_json_old",
                "eval_json_new",
            ]
        ].round(2)
    )


def eval_direction_sort_key(label):
    text = str(label)
    match = re.match(r"g=([^ ]+) / l=([^ ]+)", text)
    if not match:
        return (99, 99, text)
    return (DIRECTION_ORDER.index(match.group(1)), DIRECTION_ORDER.index(match.group(2)), text)


def plot_full_test_eval_comparison(frame=updated_eval_df, width=1150, height=None):
    plot_df = frame.dropna(
        subset=["survival_pct_old", "survival_pct_new"]
    ).copy()
    if plot_df.empty:
        raise RuntimeError("No runs have both old and continuation full-test evaluations.")
    plot_df = pd.DataFrame(
        sorted(
            plot_df.to_dict("records"),
            key=lambda row: (eval_direction_sort_key(row["direction_label"]), int(row["seed"])),
        )
    )
    plot_df["run_label"] = (
        plot_df["direction_label"].astype(str) + " s" + plot_df["seed"].astype(int).astype(str)
    )

    line_x = []
    line_y = []
    for row in plot_df.to_dict("records"):
        line_x.extend([row["survival_pct_old"], row["survival_pct_new"], None])
        line_y.extend([row["run_label"], row["run_label"], None])

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=line_x,
            y=line_y,
            mode="lines",
            name="old to continuation",
            line=dict(color="#cbd5e1", width=3),
            hoverinfo="skip",
            showlegend=False,
        )
    )
    hover = (
        "run=%{customdata[0]}<br>direction=%{customdata[1]}<br>seed=%{customdata[2]}"
        "<br>step=%{customdata[3]:.3f}M<br>survival=%{x:.2f}%"
        "<br>delta=%{customdata[4]:+.2f} pp<extra></extra>"
    )
    fig.add_trace(
        go.Scatter(
            x=plot_df["survival_pct_old"],
            y=plot_df["run_label"],
            mode="markers",
            name="old eval",
            marker=dict(color="#64748b", size=9, symbol="circle"),
            customdata=np.stack(
                [
                    plot_df["run_name"],
                    plot_df["direction_label"].astype(str),
                    plot_df["seed"],
                    plot_df["eval_step_m_old"],
                    plot_df["survival_delta_pp"],
                ],
                axis=-1,
            ),
            hovertemplate=hover,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=plot_df["survival_pct_new"],
            y=plot_df["run_label"],
            mode="markers",
            name="continuation eval",
            marker=dict(color="#f97316", size=10, symbol="diamond"),
            customdata=np.stack(
                [
                    plot_df["run_name"],
                    plot_df["direction_label"].astype(str),
                    plot_df["seed"],
                    plot_df["eval_step_m_new"],
                    plot_df["survival_delta_pp"],
                ],
                axis=-1,
            ),
            hovertemplate=hover,
        )
    )
    fig.update_layout(
        title="Old vs continuation full-test evaluation of best checkpoints",
        xaxis_title="Full-test episodic survival (%)",
        yaxis_title="Run",
        width=width,
        height=height or max(520, 34 * len(plot_df) + 160),
        margin=dict(l=170, r=40, t=80, b=60),
        hovermode="closest",
        legend=dict(orientation="h", x=0.0, y=1.06),
    )
    fig.update_xaxes(range=[0, 105])
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=plot_df["run_label"].tolist()[::-1],
    )
    return fig


eval_comparison_fig = plot_full_test_eval_comparison()
eval_comparison_fig.show()
print("done")

old evaluations: 27
continuation evaluations: 13
matched old/new evaluations: 13


,direction_label,seed,run_name,eval_step_m_old,survival_pct_old,eval_step_m_new,survival_pct_new,survival_delta_pp,step_delta_m,eval_json_old,eval_json_new
24,g=bi / l=bi,0,gs_hmd_hetero_n0_none_gbi_lbi_s0,13.11,96.23,14.76,98.40,2.17,1.66,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gbi_lbi_s0_step13105152_job3096907.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gbi_lbi_s0.json
26,g=bi / l=bi,2,gs_hmd_hetero_n0_none_gbi_lbi_s2,10.78,94.23,14.43,96.34,2.11,3.65,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gbi_lbi_s2_step10782720_job3096909.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gbi_lbi_s2.json
18,g=bi / l=a2b,0,gs_hmd_hetero_n0_none_gbi_la2b_s0,11.70,87.73,14.93,86.52,-1.21,3.23,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gbi_la2b_s0_step11695104_job3096901.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gbi_la2b_s0.json
21,g=bi / l=b2a,0,gs_hmd_hetero_n0_none_gbi_lb2a_s0,13.77,87.61,14.93,86.11,-1.51,1.16,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gbi_lb2a_s0_step13768704_job3096904.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gbi_lb2a_s0.json
6,g=a2b / l=bi,0,gs_hmd_hetero_n0_none_ga2b_lbi_s0,12.61,95.28,14.93,97.19,1.91,2.32,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_lbi_s0_step12607488_job3096889.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lbi_s0.json
0,g=a2b / l=a2b,0,gs_hmd_hetero_n0_none_ga2b_la2b_s0,4.48,69.73,14.18,69.60,-0.13,9.70,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_la2b_s0_step4478976_job3096883.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_la2b_s0.json
3,g=a2b / l=b2a,0,gs_hmd_hetero_n0_none_ga2b_lb2a_s0,12.11,98.09,14.52,98.61,0.52,2.41,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s0_step12109824_job3096886.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s0.json
4,g=a2b / l=b2a,1,gs_hmd_hetero_n0_none_ga2b_lb2a_s1,13.93,96.66,14.68,99.03,2.37,0.75,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s1_step13934592_job3096887.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_ga2b_lb2a_s1.json
16,g=b2a / l=bi,1,gs_hmd_hetero_n0_none_gb2a_lbi_s1,10.62,97.24,14.60,90.21,-7.03,3.98,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gb2a_lbi_s1_step10616832_job3096899.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gb2a_lbi_s1.json
10,g=b2a / l=a2b,1,gs_hmd_hetero_n0_none_gb2a_la2b_s1,13.60,96.92,14.85,96.96,0.04,1.24,outputs/full_test_eval/best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s1_step13602816_job3096893.json,outputs/full_test_eval/continuation_20260731/best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s1.json


done


## Plot helpers

In [8]:
def metric_title(metric):
    title = metric.replace("/", " - ").replace("_", " ")
    return title[:1].upper() + title[1:]


def metric_frame(metric=METRIC, smooth_window=SMOOTH_WINDOW):
    if metric not in history_df.columns:
        available = sorted(col for col in history_df.columns if "survival" in col)
        raise KeyError(f"Metric {metric!r} is missing. Available survival columns: {available}")
    frame = history_df.dropna(subset=[metric]).copy()
    frame["value_pct"] = pd.to_numeric(frame[metric], errors="coerce") * VALUE_SCALE
    frame = frame.dropna(subset=["value_pct"]).sort_values(["run_name", "_step"])
    if smooth_window and smooth_window > 1:
        frame["plot_value"] = frame.groupby("run_name", observed=True)["value_pct"].transform(
            lambda values: values.rolling(smooth_window, min_periods=1, center=True).mean()
        )
    else:
        frame["plot_value"] = frame["value_pct"]
    return frame


def direction_sort_key(label):
    text = str(label)
    match = re.match(r"g=([^ ]+) / l=([^ ]+)", text)
    if not match:
        return (99, 99, text)
    return (DIRECTION_ORDER.index(match.group(1)), DIRECTION_ORDER.index(match.group(2)), text)


def discontinuity_points(frame):
    points = []
    if discontinuity_df.empty:
        return pd.DataFrame(points)
    for item in discontinuity_df.to_dict("records"):
        run_name = item["run_name"]
        run_frame = frame[frame["run_name"].eq(run_name)].sort_values("_step")
        if run_frame.empty or pd.isna(item.get("discontinuity_step")):
            continue
        before = run_frame[run_frame["_step"] <= item["discontinuity_step"]]
        if before.empty:
            before = run_frame.iloc[[0]]
        point = before.iloc[-1].to_dict()
        point["discontinuity_step"] = item["discontinuity_step"]
        point["discontinuity_step_m"] = item["discontinuity_step_m"]
        point["resume_step"] = item["resume_step"]
        point["resume_step_m"] = item["resume_step_m"]
        points.append(point)
    return pd.DataFrame(points)


def aggregate_direction_curves(frame):
    return (
        frame.groupby(["direction_label", "_step", "step_m"], observed=True, as_index=False)
        .agg(
            mean_value=("plot_value", "mean"),
            std_value=("plot_value", "std"),
            seeds=("run_name", "nunique"),
        )
        .sort_values(["direction_label", "_step"])
    )

## Completed mean curves with continuation markers

In [9]:
def plot_complete_curves(
    metric=METRIC,
    smooth_window=SMOOTH_WINDOW,
    show_seed_curves=SHOW_SEED_CURVES,
    show_mean_curves=SHOW_MEAN_CURVES,
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
):
    frame = metric_frame(metric, smooth_window=smooth_window)
    aggregate = aggregate_direction_curves(frame)
    discontinuities = discontinuity_points(frame)

    fig = go.Figure()
    direction_labels = sorted(frame["direction_label"].dropna().unique(), key=direction_sort_key)

    for label in direction_labels:
        color = DIRECTION_COLORS.get(str(label), "#444444")
        group = frame[frame["direction_label"].eq(label)].sort_values("_step")
        if show_seed_curves:
            for run_name, run_frame in group.groupby("run_name", observed=True):
                seed = run_frame["seed"].iloc[0]
                fig.add_trace(
                    go.Scatter(
                        x=run_frame["step_m"],
                        y=run_frame["plot_value"],
                        mode="lines",
                        name=f"{label} seed {seed}",
                        legendgroup=str(label),
                        showlegend=False,
                        line=dict(color=color, width=1),
                        opacity=0.25,
                        hovertemplate=(
                            "run=%{customdata[0]}<br>seed=%{customdata[1]}"
                            "<br>step=%{x:.3f}M<br>survival=%{y:.2f}%<extra></extra>"
                        ),
                        customdata=np.stack(
                            [run_frame["run_name"], run_frame["seed"]], axis=-1
                        ),
                    )
                )
        if show_mean_curves:
            mean_frame = aggregate[aggregate["direction_label"].eq(label)]
            fig.add_trace(
                go.Scatter(
                    x=mean_frame["step_m"],
                    y=mean_frame["mean_value"],
                    mode="lines",
                    name=str(label),
                    legendgroup=str(label),
                    line=dict(color=color, width=3),
                    hovertemplate=(
                        f"{label}<br>step=%{{x:.3f}}M"
                        "<br>mean survival=%{y:.2f}%<br>seeds=%{customdata}<extra></extra>"
                    ),
                    customdata=mean_frame["seeds"],
                )
            )

    if not discontinuities.empty:
        fig.add_trace(
            go.Scatter(
                x=discontinuities["discontinuity_step_m"],
                y=discontinuities["plot_value"],
                mode="markers",
                name="training stopped before continuation",
                marker=dict(symbol="line-ns", size=8, color="#111827", line=dict(width=2)),
                hovertemplate=(
                    "run=%{customdata[0]}<br>direction=%{customdata[1]}"
                    "<br>seed=%{customdata[2]}<br>stopped after=%{x:.3f}M steps"
                    "<br>resumed at=%{customdata[3]:.3f}M steps"
                    "<br>survival marker=%{y:.2f}%<extra></extra>"
                ),
                customdata=np.stack(
                    [
                        discontinuities["run_name"],
                        discontinuities["direction_label"].astype(str),
                        discontinuities["seed"],
                        discontinuities["resume_step_m"],
                    ],
                    axis=-1,
                ),
            )
        )

    fig.update_layout(
        title=(
            f"gs_hmd completed curves: {metric_title(metric)} "
            f"(smooth={smooth_window})"
        ),
        xaxis_title="Environment steps (millions)",
        yaxis_title="Episodic survival (%)",
        width=width,
        height=height,
        hovermode="closest",
        legend=dict(orientation="v", x=1.02, y=1.0),
        margin=dict(l=70, r=260, t=80, b=60),
    )
    fig.update_yaxes(range=[0, 105])
    fig.add_vline(x=15.0, line_dash="dash", line_color="#9ca3af", annotation_text="15M target")
    return fig


complete_curve_fig = plot_complete_curves()
complete_curve_fig.show()

## Faceted seed curves

Use this view when you want to inspect exactly which seed was continued. The small vertical tick is the last locally cached step before the continuation segment starts.

In [10]:
def plot_seed_facets(metric=METRIC, smooth_window=SMOOTH_WINDOW, width=FIG_WIDTH, subplot_height=300):
    frame = metric_frame(metric, smooth_window=smooth_window)
    discontinuities = discontinuity_points(frame)
    direction_labels = sorted(frame["direction_label"].dropna().unique(), key=direction_sort_key)
    ncols = 3
    nrows = int(np.ceil(len(direction_labels) / ncols))
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=[str(label) for label in direction_labels],
        shared_xaxes=True,
        shared_yaxes=True,
        horizontal_spacing=0.055,
        vertical_spacing=0.10,
    )

    seed_styles = {
        0: dict(dash="solid", width=2.0),
        1: dict(dash="dash", width=2.0),
        2: dict(dash="dot", width=2.0),
    }

    for index, label in enumerate(direction_labels):
        row = index // ncols + 1
        col = index % ncols + 1
        color = DIRECTION_COLORS.get(str(label), "#444444")
        group = frame[frame["direction_label"].eq(label)].sort_values("_step")
        for run_name, run_frame in group.groupby("run_name", observed=True):
            seed = int(run_frame["seed"].iloc[0])
            style = seed_styles.get(seed, dict(dash="solid", width=1.7))
            fig.add_trace(
                go.Scatter(
                    x=run_frame["step_m"],
                    y=run_frame["plot_value"],
                    mode="lines",
                    name=f"seed {seed}",
                    legendgroup=f"seed {seed}",
                    showlegend=index == 0,
                    line=dict(color=color, **style),
                    hovertemplate=(
                        "run=%{customdata[0]}<br>seed=%{customdata[1]}"
                        "<br>step=%{x:.3f}M<br>survival=%{y:.2f}%<extra></extra>"
                    ),
                    customdata=np.stack([run_frame["run_name"], run_frame["seed"]], axis=-1),
                ),
                row=row,
                col=col,
            )

        panel_disc = discontinuities[discontinuities["direction_label"].eq(label)]
        if not panel_disc.empty:
            fig.add_trace(
                go.Scatter(
                    x=panel_disc["discontinuity_step_m"],
                    y=panel_disc["plot_value"],
                    mode="markers",
                    name="continuation marker",
                    legendgroup="continuation marker",
                    showlegend=index == 0,
                    marker=dict(symbol="line-ns", size=8, color="#111827", line=dict(width=2)),
                    hovertemplate=(
                        "run=%{customdata[0]}<br>seed=%{customdata[1]}"
                        "<br>stopped after=%{x:.3f}M steps"
                        "<br>resumed at=%{customdata[2]:.3f}M steps"
                        "<br>survival marker=%{y:.2f}%<extra></extra>"
                    ),
                    customdata=np.stack(
                        [panel_disc["run_name"], panel_disc["seed"], panel_disc["resume_step_m"]],
                        axis=-1,
                    ),
                ),
                row=row,
                col=col,
            )

    fig.update_layout(
        title=f"gs_hmd seed curves with continuation markers: {metric_title(metric)}",
        width=width,
        height=max(520, subplot_height * nrows),
        margin=dict(l=70, r=40, t=90, b=60),
        hovermode="closest",
    )
    fig.update_xaxes(title_text="Environment steps (millions)")
    fig.update_yaxes(title_text="Episodic survival (%)", range=[0, 105])
    return fig


seed_facet_fig = plot_seed_facets()
seed_facet_fig.show()

## Optional: plot another metric

Change `metric_to_plot` below to redraw the same figures for train-eval survival instead of test survival.

In [11]:
metric_to_plot = "train_eval/charts/episodic_survival"

plot_complete_curves(metric=metric_to_plot).show()
plot_seed_facets(metric=metric_to_plot).show()